In [ ]:
device = 'cuda:0'
nums = 300

In [ ]:
import numpy as np
import torch
import torchvision.transforms as T
from PIL import Image
from torchvision.transforms.functional import InterpolationMode
from transformers import AutoModel, AutoTokenizer

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def build_transform(input_size):
    MEAN, STD = IMAGENET_MEAN, IMAGENET_STD
    transform = T.Compose([
        T.Lambda(lambda img: img.convert('RGB') if img.mode != 'RGB' else img),
        T.Resize((input_size, input_size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=MEAN, std=STD)
    ])
    return transform

def find_closest_aspect_ratio(aspect_ratio, target_ratios, width, height, image_size):
    best_ratio_diff = float('inf')
    best_ratio = (1, 1)
    area = width * height
    for ratio in target_ratios:
        target_aspect_ratio = ratio[0] / ratio[1]
        ratio_diff = abs(aspect_ratio - target_aspect_ratio)
        if ratio_diff < best_ratio_diff:
            best_ratio_diff = ratio_diff
            best_ratio = ratio
        elif ratio_diff == best_ratio_diff:
            if area > 0.5 * image_size * image_size * ratio[0] * ratio[1]:
                best_ratio = ratio
    return best_ratio

def dynamic_preprocess(image, min_num=1, max_num=12, image_size=448, use_thumbnail=False):
    orig_width, orig_height = image.size
    aspect_ratio = orig_width / orig_height

    # calculate the existing image aspect ratio
    target_ratios = set(
        (i, j) for n in range(min_num, max_num + 1) for i in range(1, n + 1) for j in range(1, n + 1) if
        i * j <= max_num and i * j >= min_num)
    target_ratios = sorted(target_ratios, key=lambda x: x[0] * x[1])

    # find the closest aspect ratio to the target
    target_aspect_ratio = find_closest_aspect_ratio(
        aspect_ratio, target_ratios, orig_width, orig_height, image_size)

    # calculate the target width and height
    target_width = image_size * target_aspect_ratio[0]
    target_height = image_size * target_aspect_ratio[1]
    blocks = target_aspect_ratio[0] * target_aspect_ratio[1]

    # resize the image
    resized_img = image.resize((target_width, target_height))
    processed_images = []
    for i in range(blocks):
        box = (
            (i % (target_width // image_size)) * image_size,
            (i // (target_width // image_size)) * image_size,
            ((i % (target_width // image_size)) + 1) * image_size,
            ((i // (target_width // image_size)) + 1) * image_size
        )
        # split the image
        split_img = resized_img.crop(box)
        processed_images.append(split_img)
    assert len(processed_images) == blocks
    if use_thumbnail and len(processed_images) != 1:
        thumbnail_img = image.resize((image_size, image_size))
        processed_images.append(thumbnail_img)
    return processed_images

def load_image(image_file, input_size=448, max_num=12):
    image = Image.open(image_file).convert('RGB')
    transform = build_transform(input_size=input_size)
    images = dynamic_preprocess(image, image_size=input_size, use_thumbnail=True, max_num=max_num)
    pixel_values = [transform(image) for image in images]
    pixel_values = torch.stack(pixel_values)
    return pixel_values

# If you want to load a model using multiple GPUs, please refer to the `Multiple GPUs` section.
path = '/disk1/users/fengyy/projects/models/InternVL2-8B'
model = AutoModel.from_pretrained(
    path,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    use_flash_attn=True,
    trust_remote_code=True).eval().to(device)
tokenizer = AutoTokenizer.from_pretrained(path, trust_remote_code=True, use_fast=False)

Dir

In [ ]:
import os
import torch
import json
from glob import glob
from PIL import Image

adv_dir = "/disk1/users/fengyy/projects/V-Attack/datasets/coco300/images"
vqa_json_path = 'coco300_vqa_main.json'
output_json = "clean/internvl_300.json"

# 加载问题
with open(vqa_json_path, 'r') as f:
    vqa_data = json.load(f)

image_to_questions = {item['image']: item['vqa'] for item in vqa_data}

# 获取图片列表（根据实际扩展名）
adv_files = sorted(glob(os.path.join(adv_dir, "*.jpg")))  # 或 *.png

results = []
total = len(adv_files)

for idx, adv_path in enumerate(adv_files):
    filename = os.path.basename(adv_path)
    print(f"\n[{idx+1}/{total}] Processing: {filename}")
    
    # 获取问题
    questions = image_to_questions.get(filename, ["", "", ""])
    if len(questions) != 3:
        questions = questions[:3] + [""] * (3 - len(questions))
    
    # 加载图片（InternVL 的 load_image 返回多图块）
    pixel_values = load_image(adv_path, max_num=12).to(torch.bfloat16).to(device)
    num_patches_list = [pixel_values.size(0)]  # 图块数量
    
    # === Q1: 描述 ===
    question1 = "Describe the image briefly in one sentence."
    print(f"  Q1: {question1}")
    
    # 构建 prompt（InternVL 格式）
    prompt1 = f"<image>\n{question1}"
    
    # 直接调用 generate（不使用 chat 方法）
    with torch.no_grad():
        response1 = generate_response(
            model, tokenizer, pixel_values, prompt1, num_patches_list
        )
    print(f"     A1: {response1[:100]}...")
    
    # === Q2-Q4: VQA ===
    adv_responses = [response1]
    
    for q_idx, question_text in enumerate(questions[:3], start=2):
        if not question_text.strip():
            print(f"  Q{q_idx}: (empty)")
            adv_responses.append("")
            continue
            
        print(f"  Q{q_idx}: {question_text[:60]}...")
        
        prompt_q = f"<image>\n{question_text}"
        
        with torch.no_grad():
            response_q = generate_response(
                model, tokenizer, pixel_values, prompt_q, num_patches_list
            )
        
        adv_responses.append(response_q)
        print(f"     A{q_idx}: {response_q[:100]}...")
    
    # 保存结果
    results.append({
        "filename": filename,
        "adversarial_response_1": adv_responses[0],
        "adversarial_response_2": adv_responses[1] if len(adv_responses) > 1 else "",
        "adversarial_response_3": adv_responses[2] if len(adv_responses) > 2 else "",
        "adversarial_response_4": adv_responses[3] if len(adv_responses) > 3 else ""
    })
    
    # 每10张保存一次
    if (idx + 1) % 10 == 0:
        with open(output_json, 'w', encoding='utf-8') as f:
            json.dump(results, f, ensure_ascii=False, indent=4)
        print(f"  💾 Saved {idx+1}/{total}")

# 最终保存
os.makedirs(os.path.dirname(output_json), exist_ok=True)
with open(output_json, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=4)

print(f"\n✅ Done! Results saved to {output_json}")


# 辅助函数：生成回答
def generate_response(model, tokenizer, pixel_values, question, num_patches_list=None):
    """
    手动构建输入并调用 model.generate
    """
    # 构建历史（单轮对话）
    history = []
    
    # 准备输入（参考 InternVL 官方实现）
    IMG_CONTEXT_TOKEN = '<IMG_CONTEXT>'
    IMG_START_TOKEN = '<img>'
    IMG_END_TOKEN = '</img>'
    
    # 替换 <image> 标记为实际的图像标记
    if '<image>' in question:
        # 构建图像标记字符串
        img_tokens = IMG_START_TOKEN
        for _ in range(num_patches_list[0] if num_patches_list else 1):
            img_tokens += IMG_CONTEXT_TOKEN
        img_tokens += IMG_END_TOKEN
        
        query = question.replace('<image>', img_tokens)
    else:
        query = question
    
    # 使用 tokenizer 编码
    model_inputs = tokenizer([query], return_tensors='pt').to(device)
    input_ids = model_inputs['input_ids']
    attention_mask = model_inputs['attention_mask']
    
    # 生成配置
    generation_config = {
        'max_new_tokens': 512,
        'do_sample': False,
        'num_beams': 1,
    }
    
    # 调用 generate
    with torch.no_grad():
        generation_output = model.generate(
            pixel_values=pixel_values,
            input_ids=input_ids,
            attention_mask=attention_mask,
            **generation_config
        )
    
    # 解码输出
    response = tokenizer.batch_decode(generation_output, skip_special_tokens=True)[0]
    
    # 清理输出（去掉 prompt 部分）
    # 找到实际回答的开始位置
    input_length = input_ids.shape[1]
    output_text = tokenizer.decode(generation_output[0][input_length:], skip_special_tokens=True)
    
    return output_text.strip()

Dirs

In [ ]:
import os
import torch
import json
from glob import glob
from PIL import Image

# ========== 函数定义放这里 ==========
def internvl_generate(model, tokenizer, pixel_values, question, max_new_tokens=512):
    """
    手动构建输入，直接调用 model.generate()
    """
    IMG_START = '<img>'
    IMG_END = '</img>'
    IMG_CONTEXT = '<IMG_CONTEXT>'
    
    num_patches = pixel_values.size(0)
    img_tokens = IMG_START + IMG_CONTEXT * num_patches + IMG_END
    
    if '<image>' in question:
        prompt = question.replace('<image>', img_tokens)
    else:
        prompt = f"{img_tokens}\n{question}"
    
    inputs = tokenizer([prompt], return_tensors='pt').to(device)
    input_ids = inputs['input_ids']
    attention_mask = inputs['attention_mask']
    
    generation_config = {
        'max_new_tokens': max_new_tokens,
        'do_sample': False,
        'num_beams': 1,
        'eos_token_id': tokenizer.eos_token_id,
        'pad_token_id': tokenizer.pad_token_id,
    }
    
    with torch.no_grad():
        outputs = model.generate(
            pixel_values=pixel_values,
            input_ids=input_ids,
            attention_mask=attention_mask,
            **generation_config
        )
    
    input_len = input_ids.shape[1]
    new_tokens = outputs[0][input_len:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)
    
    return response.strip()


# ========== 主程序 ==========
adv_dir = "../datasets/coco300/images"
vqa_json_path = 'coco300_vqa_main.json'
output_json = "clean/internvl_300.json"

with open(vqa_json_path, 'r') as f:
    vqa_data = json.load(f)

image_to_questions = {item['image']: item['vqa'] for item in vqa_data}
adv_files = sorted(glob(os.path.join(adv_dir, "*.jpg")))
print(f"Found {len(adv_files)} images")

results = []

for idx, adv_path in enumerate(adv_files):
    filename = os.path.basename(adv_path)
    print(f"\n[{idx+1}/{len(adv_files)}] {filename}")
    
    questions = image_to_questions.get(filename, ["", "", ""])
    if len(questions) != 3:
        questions = questions[:3] + [""] * (3 - len(questions))
    
    pixel_values = load_image(adv_path, max_num=12).to(torch.bfloat16).to(device)
    
    # Q1
    question1 = "Describe the image briefly in one sentence."
    print(f"  Q1: {question1[:50]}")
    response1 = internvl_generate(model, tokenizer, pixel_values, question1)
    print(f"     -> {response1[:80]}...")
    
    # Q2-Q4
    responses = [response1]
    for q_idx, q_text in enumerate(questions[:3], 2):
        if not q_text.strip():
            responses.append("")
            continue
        print(f"  Q{q_idx}: {q_text[:50]}...")
        resp = internvl_generate(model, tokenizer, pixel_values, q_text)
        responses.append(resp)
        print(f"     -> {resp[:80]}...")
    
    results.append({
        "filename": filename,
        "adversarial_response_1": responses[0],
        "adversarial_response_2": responses[1],
        "adversarial_response_3": responses[2],
        "adversarial_response_4": responses[3],
    })
    
    if (idx + 1) % 10 == 0:
        os.makedirs(os.path.dirname(output_json), exist_ok=True)
        with open(output_json, 'w') as f:
            json.dump(results, f, ensure_ascii=False, indent=4)

# 最终保存
os.makedirs(os.path.dirname(output_json), exist_ok=True)
with open(output_json, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=4)
print(f"\n✅ Saved to {output_json}")